# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Корнілов Артем'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-31'
НОМЕР        = 10       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [2]:
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json', encoding='utf-8'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 10   ·   Корнілов Артем   ·   ШІ-31

Ваші вісім стовпців:
   1. Телефон
   2. Супутній
   3. ступінь КТ
   4. Поступлення
   5. КТ
   6. АТ пост
   7. Лім1
   8. Цукор1

Ваша пара для аналізу зв’язку: Вік та помер/жив

Ваше додаткове питання:
   Чому для стовпця «ступінь КТ» медіана має сенс, а середнє — ні? Наведіть конкретний приклад із вашими даними.


In [4]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,Телефон,Супутній,ступінь КТ,Поступлення,КТ,АТ пост,Лім1,Цукор1
0,0958469788 дружина Ганна,"ІХС, КС, СН1, НВ ФК2",сердній,2022-02-24,20-30%,140/100,30.0,7.6
1,"0671778486, 0994887663 дружина Оксана","ЦД, ІІ тип",середній,2022-02-22,30-35%,130/80,9.8,19.9
2,"0958768045, чоловік Степан 0951784726","ІХС. КС, ГХІІ, ГЛШ СНІІа, порушення толерантності до глю...",критичний,2021-12-30,70-80%,150/100,12.0,6.2
3,0959819331. дочка Валентина 0960845205,"ЦД, ІІ тип, ІХС, Диф кардіоск, стенокар напруги, синусов...",критичний,2021-12-24,0.8,170/110,8.0,15.5
4,"0952294866, дочка Дарина 0678360161",ІХС ДК ГХІІ-ІІІ СНІІа ГЛШст3,NaN,2022-02-12,NaN,160/100,5.0,4.4
5,0959922983 сестра Софія,"ІХС ДК ГХІІ ГЛШ СНІІ ФК2, ЦД ІІ",середній,2021-12-15,30-40%,140/90,6.0,33.8
6,"0998073702, дружина Наталія 0998382226","ІХС ДК ГХ ІІа ФК3, ЦД вперше виявлений",важкий,2022-04-23,60-65%,130/80,11.0,5.9
7,"0637258771, син Сергій 0968413301","ІХС ДК ГХІІ ГЛШ СНІІа ФК3, полімедик алергія,ерозивий г/...",легкий,2022-05-18,до 17%,140/80,17.0,5.7


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [5]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Телефон      120 non-null    object        
 1   Супутній     115 non-null    str           
 2   ступінь КТ   42 non-null     str           
 3   Поступлення  120 non-null    datetime64[us]
 4   КТ           44 non-null     object        
 5   АТ пост      120 non-null    str           
 6   Лім1         117 non-null    float64       
 7   Цукор1       118 non-null    float64       
dtypes: datetime64[us](1), float64(2), object(2), str(3)
memory usage: 7.7+ KB


In [6]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- Телефон
    непорожніх 120 з 121 | унікальних 120 | типи: str×97, int×23
    приклади: ["'0958469788 дружина Ганна'", "'0671778486, 0994887663 дружина Оксана'", "'0958768045, чоловік Степан 0951784726'", "'0959819331. дочка Валентина 0960845205'"]

--- Супутній
    непорожніх 115 з 121 | унікальних 91 | типи: str×115
    приклади: ["'ІХС, КС, СН1, НВ ФК2'", "'ЦД, ІІ тип'", "'ІХС. КС, ГХІІ, ГЛШ СНІІа, порушення толерантності до глюкози'", "'ЦД, ІІ тип, ІХС, Диф кардіоск, стенокар напруги, синусова тахік, ГХІІ, СНІІв '"]

--- ступінь КТ
    непорожніх 42 з 121 | унікальних 12 | типи: str×42
    приклади: ["'сердній'", "'середній'", "'критичний'", "'важкий'"]

--- Поступлення
    непорожніх 120 з 121 | унікальних 93 | типи: Timestamp×120
    приклади: ["Timestamp('2022-02-24 00:00:00')", "Timestamp('2022-02-22 00:00:00')", "Timestamp('2021-12-30 00:00:00')", "Timestamp('2021-12-24 00:00:00')"]

--- КТ
    непорожніх 44 з 121 | унікальних 26 | типи: float×27, str×17
    приклади: ["'20

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [7]:
ПАСПОРТ = {
    'Телефон': ('ідентифікатор',
                'Номер телефону лише вказує, кому дзвонити; це не величина: середнє, різниця чи '
                'порядок двох номерів нічого не означають, а в клітинках ще й імена родичів.'),
    'Супутній': ('номінальна',
                 'Перелік супутніх діагнозів (ІХС, ГХ, ЦД, СН…) — категорії, які можна лише '
                 'порівняти на рівність; «ІХС» не більша й не менша за «ЦД», а в одній '
                 'клітинці буває кілька міток.'),
    'ступінь КТ': ('порядкова',
                   'Рівні легкий < середній < важкий < вкрай важкий < критичний мають природний '
                   'порядок, але ніхто не стверджує, що крок «легкий→середній» дорівнює кроку '
                   '«важкий→критичний».'),
    'Поступлення': ('інтервальна',
                    'Дата госпіталізації: різниця двох дат у днях має сенс, а нуль календаря '
                    'умовний, тому «удвічі пізніша дата» чи відсоток від дати — безглуздя.'),
    'КТ': ('абсолютна',
           'Частка ураженої тканини легень (0.3 = 30 %): природний нуль — ураження немає, '
           'природна одиниця — уражена вся легеня; «удвічі більше ураження» має сенс.'),
    'АТ пост': ('відношень',
                'Систолічний і діастолічний тиск у мм рт. ст.: нуль тиску природний, 160 справді '
                'вдвічі більше за 80; але рядок «130/80» — це дві величини, їх треба розділити.'),
    'Лім1': ('абсолютна',
             'Лімфоцити в лейкоцитарній формулі у відсотках — частка від усіх лейкоцитів, '
             'тож природні і нуль (0 %), і одиниця (100 %).'),
    'Цукор1': ('відношень',
               'Глюкоза крові в ммоль/л: нуль означає відсутність глюкози, 14 ммоль/л справді '
               'вдвічі більше за 7, але одиниця умовна (можна мг/дл), тому шкала не абсолютна.'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,Телефон,ідентифікатор,"Номер телефону лише вказує, кому дзвонити; це не величин..."
1,Супутній,номінальна,"Перелік супутніх діагнозів (ІХС, ГХ, ЦД, СН…) — категорі..."
2,ступінь КТ,порядкова,Рівні легкий < середній < важкий < вкрай важкий < критич...
3,Поступлення,інтервальна,"Дата госпіталізації: різниця двох дат у днях має сенс, а..."
4,КТ,абсолютна,Частка ураженої тканини легень (0.3 = 30 %): природний н...
5,АТ пост,відношень,Систолічний і діастолічний тиск у мм рт. ст.: нуль тиску...
6,Лім1,абсолютна,Лімфоцити в лейкоцитарній формулі у відсотках — частка в...
7,Цукор1,відношень,Глюкоза крові в ммоль/л: нуль означає відсутність глюкоз...


In [8]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


### Приведення стовпців до їхньої шкали

`info()` показав, що pandas бачить шкали неправильно: `ступінь КТ` — 12 різних рядків замість
п'яти рівнів, `КТ` — суміш часток (0.3) і рядків-інтервалів («25-30%»), `АТ пост` — рядок
«130/80» з двох чисел, `Телефон` — то число, то текст з іменем. Перш ніж рахувати статистики,
кожен стовпець перетворюємо так, щоб його шкала стала видимою для pandas.

In [9]:
CT_ORDER = ['легкий', 'середній', 'важкий', 'вкрай важкий', 'критичний']
CT_SYNONYMS = {'легка': 'легкий', 'сердній': 'середній',
               'серед ступ': 'середній', 'серед ступеня': 'середній'}


def parse_ct_severity(raw):
    """Чистить написання і задає порядок рівнів вручну, а не за алфавітом."""
    s = raw.astype('string').str.strip().replace(CT_SYNONYMS)
    return pd.Series(pd.Categorical(s, categories=CT_ORDER, ordered=True),
                     index=raw.index, name=raw.name)


def parse_ct_fraction(v):
    """0.3 -> 0.3;  '25-30%' -> 0.275 (середина інтервалу);  'до 17%' -> 0.17;  пробіли -> NaN."""
    if isinstance(v, (int, float)):
        return float(v)
    numbers = re.findall(r'\d+(?:[.,]\d+)?', str(v))
    if not numbers:
        return np.nan
    return np.mean([float(number.replace(',', '.')) for number in numbers]) / 100


def parse_blood_pressure(raw):
    """'130/80' -> два числові стовпці: систолічний і діастолічний тиск."""
    parts = raw.astype('string').str.extract(r'(\d+)\s*/\s*(\d+)').astype(float)
    parts.columns = ['САТ', 'ДАТ']
    return parts


blood_pressure = parse_blood_pressure(df['АТ пост'])
CLEAN_DATA = {
    'Телефон':       df['Телефон'].astype('string'),
    'Супутній':      df['Супутній'].astype('string').str.strip(),
    'ступінь КТ':    parse_ct_severity(df['ступінь КТ']),
    'Поступлення':   pd.to_datetime(df['Поступлення'], errors='coerce'),
    'КТ':            df['КТ'].map(parse_ct_fraction),
    'АТ пост (САТ)': blood_pressure['САТ'],
    'АТ пост (ДАТ)': blood_pressure['ДАТ'],
    'Лім1':          pd.to_numeric(df['Лім1'], errors='coerce'),
    'Цукор1':        pd.to_numeric(df['Цукор1'], errors='coerce'),
}
# які очищені ряди відповідають кожному стовпцю варіанта (АТ пост -> два)
COMPONENTS = {c: [k for k in CLEAN_DATA if k == c or k.startswith(c + ' (')] for c in МОЇ}

print('ступінь КТ: було', df['ступінь КТ'].dropna().nunique(), 'написань, стало',
      CLEAN_DATA['ступінь КТ'].nunique(), 'рівнів:')
print(CLEAN_DATA['ступінь КТ'].value_counts(sort=False).to_string())
print()
print('КТ: числових значень у сирому стовпці', pd.to_numeric(df['КТ'], errors='coerce').notna().sum(),
      '| після розбору інтервалів', CLEAN_DATA['КТ'].notna().sum(), 'з', df['КТ'].notna().sum())
print('АТ пост: розібрано', blood_pressure.notna().all(axis=1).sum(), 'з', df['АТ пост'].notna().sum())

ступінь КТ: було 12 написань, стало 5 рівнів:
ступінь КТ
легкий           7
середній        18
важкий          10
вкрай важкий     4
критичний        3

КТ: числових значень у сирому стовпці 27 | після розбору інтервалів 43 з 44
АТ пост: розібрано 120 з 120


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [10]:
def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = CLEAN_DATA[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        р['повторів'] = int(з.duplicated().sum())     # єдине змістовне: чи немає дублів
        return р
    # усі шкали: мода і частоти
    р['мода'] = з.mode().iloc[0]
    р['частота моди'] = int((з == р['мода']).sum())
    if ш == 'номінальна':
        р['топ-3 частоти'] = з.value_counts().head(3).to_dict()
        return р
    # порядкова: медіана й квартилі — на кодах заданого порядку, результат назад у рівень
    if ш == 'порядкова':
        codes = pd.Series(з.cat.codes)
        for назва, q in (('Q1', .25), ('медіана', .5), ('Q3', .75)):
            р[назва] = з.cat.categories[int(codes.quantile(q, interpolation='lower'))]
        р['частоти за порядком'] = з.value_counts(sort=False).to_dict()
        return р
    р['медіана'] = з.median()
    р['Q1'], р['Q3'] = з.quantile(.25), з.quantile(.75)
    р['середнє'] = з.mean()
    р['ст. відхилення'] = з.std()
    if ш == 'інтервальна':
        return р
    # відношень і абсолютна: є природний нуль, тому відношення мають сенс
    р['коеф. варіації, %'] = 100 * з.std() / з.mean()
    if (з > 0).all():
        р['геом. середнє'] = float(np.exp(np.log(з).mean()))
    return р


def format_value(v):
    if isinstance(v, pd.Timestamp):
        return str(v.date())
    if isinstance(v, pd.Timedelta):
        return f'{v / pd.Timedelta(days=1):.1f} дн.'
    if isinstance(v, (float, np.floating)):
        return round(float(v), 3)
    return v


for c in МОЇ:
    ш = ПАСПОРТ[c][0]
    for k in COMPONENTS[c]:
        print(f'--- {k}  ({ш})')
        for назва, v in допустима_статистика(k, ш).items():
            print(f'    {назва:20} {format_value(v)}')
        print()

--- Телефон  (ідентифікатор)
    непорожніх           120
    унікальних           120
    повторів             0

--- Супутній  (номінальна)
    непорожніх           115
    унікальних           90
    мода                 ІХС ДК ГХст2
    частота моди         7
    топ-3 частоти        {'ІХС ДК ГХст2': 7, 'ІХС.ДК. ГХ ІІ': 5, 'ІХС ДК ГХст2.': 4}

--- ступінь КТ  (порядкова)
    непорожніх           42
    унікальних           5
    мода                 середній
    частота моди         18
    Q1                   середній
    медіана              середній
    Q3                   важкий
    частоти за порядком  {'легкий': 7, 'середній': 18, 'важкий': 10, 'вкрай важкий': 4, 'критичний': 3}

--- Поступлення  (інтервальна)
    непорожніх           120
    унікальних           93
    мода                 2022-04-02
    частота моди         4
    медіана              2022-02-27
    Q1                   2021-11-05
    Q3                   2022-04-07
    середнє              2022-01-12
    с

Для `Супутній` мода цілого рядка («ІХС ДК ГХст2») мало що каже: у клітинці перелік діагнозів.
Тому для номінальної змінної з кількома мітками рахуємо частоти **окремих діагнозів** —
це все ще лише частоти, тобто в межах номінальної шкали.

In [11]:
DIAGNOSES = {'ІХС': r'ІХС', 'ГХ': r'ГХ', 'ЦД': r'ЦД|ЦЖ|цукр', 'СН': r'СН'}   # «ЦЖ» — описка в даних

comorbidities = CLEAN_DATA['Супутній']
diagnosis_counts = pd.Series(
    {diagnosis: int(comorbidities.str.contains(ш, regex=True).fillna(False).sum()) for diagnosis, ш in DIAGNOSES.items()})
print('Пацієнтів із заповненим полем:', comorbidities.notna().sum(), 'з', len(df))
print((diagnosis_counts.to_frame('пацієнтів')
       .assign(**{'% від заповнених': lambda t: (100 * t['пацієнтів'] / comorbidities.notna().sum()).round(1)})
       ).to_string())

Пацієнтів із заповненим полем: 115 з 121
     пацієнтів  % від заповнених
ІХС         98              85.2
ГХ          70              60.9
ЦД          29              25.2
СН          33              28.7


---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [13]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Телефон,120,120,0958469788 дружина Ганна,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Супутній,115,91,ІХС ДК ГХст2,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ступінь КТ,42,12,середній,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Поступлення,120,NaN,NaN,NaN,2022-01-12 07:24:00,2021-02-25 00:00:00,2021-11-05 06:00:00,2022-02-27 12:00:00,2022-04-07 00:00:00,2022-06-03 00:00:00,NaN
КТ,44.0,26.0,0.3,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
АТ пост,120,23,130/80,33,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Лім1,117.0,NaN,NaN,NaN,14.281197,1.0,7.0,12.0,18.0,57.0,10.497667
Цукор1,118.0,NaN,NaN,NaN,7.982966,3.2,5.1,6.9,9.0,33.8,4.725707


In [14]:
summary = df[МОЇ].describe(include='all').T

numeric_phones = pd.to_numeric(df['Телефон'], errors='coerce')
alphabetical_codes = {label: i for i, label in enumerate(sorted(df['ступінь КТ'].dropna().unique()))}   # як LabelEncoder
raw_ct = pd.to_numeric(df['КТ'], errors='coerce')
days_1970 = (CLEAN_DATA['Поступлення'] - pd.Timestamp('1970-01-01')).dt.days
days_2021 = (CLEAN_DATA['Поступлення'] - pd.Timestamp('2021-01-01')).dt.days
cv_1970 = 100 * days_1970.std() / days_1970.mean()
cv_2021 = 100 * days_2021.std() / days_2021.mean()

БЕЗГЛУЗДІ = [
    ('Телефон', f'середнє після pd.to_numeric ({numeric_phones.notna().sum()} з {df["Телефон"].notna().sum()})',
     round(numeric_phones.mean()),
     'Телефон — ідентифікатор: «середній номер» не належить нікому. До того ж pandas узяв лише '
     'клітинки, де Excel зберіг номер як число (і загубив провідний нуль), а рядки з іменами відкинув.'),
    ('Телефон', 'top / freq у describe',
     f"{summary.loc['Телефон', 'top']} × {summary.loc['Телефон', 'freq']}",
     'Усі значення унікальні, тож «найчастіше значення» — просто перший рядок, що трапився; '
     'мода ідентифікатора нічого не каже.'),
    ('ступінь КТ', 'unique у describe', summary.loc['ступінь КТ', 'unique'],
     f"pandas рахує різні рядки, а не рівні тяжкості: «середній», «сердній», «серед ступ», «легкий » з "
     f"пробілом… Після чистки рівнів лише {CLEAN_DATA['ступінь КТ'].nunique()} — число 12 вимірює "
     f"неохайність введення."),
    ('ступінь КТ', 'середнє алфавітних кодів', round(df['ступінь КТ'].map(alphabetical_codes).mean(), 2),
     'Коди присвоєно за алфавітом («важкий»=0, «легкий»=6, «середній»=11), а не за тяжкістю; середнє '
     'таких кодів — довільне число, що зміниться від першої ж нової описки.'),
    ('КТ', f'середнє pd.to_numeric ({raw_ct.notna().sum()} з {df["КТ"].notna().sum()})',
     round(raw_ct.mean(), 3),
     f'Інтервали на кшталт «70-80%» мовчки стали NaN і випали — а це здебільшого пацієнти з більшим '
     f'ураженням. Середнє зміщене; після розбору всіх записів воно {CLEAN_DATA["КТ"].mean():.3f}.'),
    ('Поступлення', 'коефіцієнт варіації дат, %', round(cv_1970, 3),
     f'Дата — інтервальна шкала з умовним нулем (01.01.1970). CV ділить розкид на відстань від цього '
     f'нуля: якщо рахувати дні від 01.01.2021, той самий розкид дасть {cv_2021:.1f} %.'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Телефон,середнє після pd.to_numeric (23 з 120),758798950,Телефон — ідентифікатор: «середній номер» не належить ні...
1,Телефон,top / freq у describe,0958469788 дружина Ганна × 1,"Усі значення унікальні, тож «найчастіше значення» — прос..."
2,ступінь КТ,unique у describe,12,"pandas рахує різні рядки, а не рівні тяжкості: «середній..."
3,ступінь КТ,середнє алфавітних кодів,5.9,"Коди присвоєно за алфавітом («важкий»=0, «легкий»=6, «се..."
4,КТ,середнє pd.to_numeric (27 з 44),0.396,Інтервали на кшталт «70-80%» мовчки стали NaN і випали —...
5,Поступлення,"коефіцієнт варіації дат, %",0.595,Дата — інтервальна шкала з умовним нулем (01.01.1970). C...


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [15]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Вік × помер/жив

Вік: шкала — не у вашому списку
помер/жив: шкала — не у вашому списку

   Вік  помер/жив
0   55          1
1   43          2
2   54          2
3   68          2
4   65          2
5   66          1


In [16]:
ОБРАНА_МІРА = 'точковий бісеріальний коефіцієнт кореляції'
ЧОМУ_САМЕ_ВОНА = ('Вік — шкала відношень (роки, природний нуль), а «помер/жив» — бінарна номінальна '
                  'змінна: 1 і 2 тут лише мітки. Для пари «відношень × бінарна номінальна» підходить '
                  'точковий бісеріальний коефіцієнт')

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

age = df[A]
deceased = (df[B] == 1).astype(int)          # кодування задаємо явно: 1 = помер, 0 = вижив
r_pb, p_pb = stats.pointbiserialr(deceased, age)

print(f'Точковий бісеріальний r = {r_pb:.3f},  p = {p_pb:.1e},  r² = {r_pb**2:.2f}')
print()
groups = df.groupby(deceased.map({1: 'помер', 0: 'вижив'}))[A].agg(['count', 'mean', 'median', 'std']).round(1)
print(groups.to_string())
print()
print(f'Померлі в середньому старші на {groups.loc["помер", "mean"] - groups.loc["вижив", "mean"]:.1f} року.')
print(f'r_pb > 0 означає: що старший пацієнт, то частіше він у групі «помер» (бо помер = 1).')
print(f'Вік пояснює лише ≈{100 * r_pb**2:.0f} % розкиду результату — зв’язок помірний, але статистично значущий.')

Точковий бісеріальний r = 0.449,  p = 2.4e-07,  r² = 0.20

           count  mean  median   std
помер/жив                           
вижив         60  59.2    61.0  10.4
помер         61  70.1    71.0  11.4

Померлі в середньому старші на 10.9 року.
r_pb > 0 означає: що старший пацієнт, то частіше він у групі «помер» (бо помер = 1).
Вік пояснює лише ≈20 % розкиду результату — зв’язок помірний, але статистично значущий.


In [17]:
# Тепер — контрприклад: застосуйте кореляцію Пірсона незалежно від шкал
# і поясніть, чому отримане число не можна використовувати.

# 1) Для бінарної пари Пірсон арифметично ДОРІВНЮЄ r_pb — але знак залежить від того,
#    як закодовано мітки: з вихідними 1 = помер, 2 = жив отримуємо протилежне число.
print(f'Пірсон: Вік × помер/жив (1/2 як є)  = {age.corr(df[B]):+.3f}')
print(f'Пірсон: Вік × помер (0/1)            = {age.corr(deceased):+.3f}   (= r_pb)')
print()

# 2) Там, де Пірсон справді недопустимий: Вік × порядкова «ступінь КТ».
severity = CLEAN_DATA['ступінь КТ']
ordinal_codes = severity.cat.codes.where(severity.notna())                     # 0,1,2,3,4
power_codes = ordinal_codes.map({0: 1, 1: 2, 2: 4, 3: 8, 4: 16})  # той самий порядок, інші відстані
alphabetical_ct_codes = df['ступінь КТ'].map(alphabetical_codes)                          # як зробив би LabelEncoder

table = pd.DataFrame({
    'Пірсон': [age.corr(ordinal_codes), age.corr(power_codes), age.corr(alphabetical_ct_codes)],
    'Спірмен': [age.corr(ordinal_codes, method='spearman'), age.corr(power_codes, method='spearman'),
                age.corr(alphabetical_ct_codes, method='spearman')],
}, index=['коди 0..4 за тяжкістю', 'коди 1,2,4,8,16 (той самий порядок)', 'коди за алфавітом'])
print('Вік × ступінь КТ:')
print(table.round(3).to_string())
print()

Пірсон: Вік × помер/жив (1/2 як є)  = -0.449
Пірсон: Вік × помер (0/1)            = +0.449   (= r_pb)

Вік × ступінь КТ:
                                     Пірсон  Спірмен
коди 0..4 за тяжкістю                 0.313    0.296
коди 1,2,4,8,16 (той самий порядок)   0.263    0.296
коди за алфавітом                    -0.090   -0.041

Пірсон від Вік × Телефон (номери як числа): -0.238


**Чому число Пірсона тут не можна інтерпретувати.** Пірсон вимірює *лінійний* зв'язок, тобто
вважає відстані між кодами справжніми відстанями. Для порядкової шкали відстаней немає: коди
0..4 і 1,2,4,8,16 зберігають однаковий порядок тяжкості, але Пірсон дає різні значення, а при
алфавітних кодах навіть змінює знак. Спірмен для двох «правильних» кодувань однаковий, бо
залежить лише від порядку. Для бінарної змінної Пірсон технічно збігається з r_pb, але знак
залежить від довільного вибору, кого кодувати більшим числом, — тому кодування треба задавати
явно й інтерпретувати як «точковий бісеріальний». Кореляція з телефоном — просто шум від
ідентифікатора.

---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [18]:
ПІДГОТОВКА = {
    'Телефон':     'у модель не брати: ідентифікатор без інформації про стан пацієнта (і персональні дані)',
    'Супутній':    'розібрати на окремі діагнози (ІХС, ГХ, ЦД, СН) і закодувати multi-hot 0/1 — '
                   'one-hot без порядку, бо в клітинці кілька міток',
    'ступінь КТ':  'вичистити написання й перевести в цілі коди 0…4 за порядком, заданим вручну '
                   '(легкий→критичний); пропуски (79 з 121) позначити окремою ознакою',
    'Поступлення': 'дату перевести в тривалість — дні від першої госпіталізації в наборі, '
                   'потім стандартизувати',
    'КТ':          'розібрати рядки-інтервали («25-30%» → 0.275) у частку 0…1; вона вже нормована, '
                   'додатково масштабувати не треба',
    'АТ пост':     'розділити «130/80» на систолічний і діастолічний тиск і стандартизувати кожен',
    'Лім1':        'частка у %, min-max нормалізація після перевірки варіативності (max ≠ min)',
    'Цукор1':      'сильна правостороння асиметрія (до 33.8 ммоль/л): логарифмувати, потім стандартизувати',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,Телефон,ідентифікатор,у модель не брати: ідентифікатор без інформації про стан...
1,Супутній,номінальна,"розібрати на окремі діагнози (ІХС, ГХ, ЦД, СН) і закодув..."
2,ступінь КТ,порядкова,вичистити написання й перевести в цілі коди 0…4 за поряд...
3,Поступлення,інтервальна,дату перевести в тривалість — дні від першої госпіталіза...
4,КТ,абсолютна,розібрати рядки-інтервали («25-30%» → 0.275) у частку 0…...
5,АТ пост,відношень,розділити «130/80» на систолічний і діастолічний тиск і ...
6,Лім1,абсолютна,"частка у %, min-max нормалізація після перевірки варіати..."
7,Цукор1,відношень,сильна правостороння асиметрія (до 33.8 ммоль/л): логари...


In [20]:
def standardize(s):
    return (s - s.mean()) / s.std()


def minmax(s):
    value_range = s.max() - s.min()
    if not value_range > 0:
        raise ValueError(f'{s.name}: max = min, нормалізувати неможливо')
    return (s - s.min()) / value_range


X = pd.DataFrame(index=df.index)

# 1) номінальна — Супутній: multi-hot за діагнозами, жодного порядку між ними
for diagnosis, pattern in DIAGNOSES.items():
    X[f'супутній_{diagnosis}'] = CLEAN_DATA['Супутній'].str.contains(pattern, regex=True).fillna(False).astype(int)

# 2) порядкова — ступінь КТ: коди з порядком, заданим вручну в ПОРЯДОК_КТ
X['ступінь_КТ'] = CLEAN_DATA['ступінь КТ'].cat.codes.replace(-1, np.nan)
X['ступінь_КТ_пропуск'] = CLEAN_DATA['ступінь КТ'].isna().astype(int)

# 3) інтервальна — Поступлення: дата -> тривалість у днях -> стандартизація
days = (CLEAN_DATA['Поступлення'] - CLEAN_DATA['Поступлення'].min()).dt.days
X['поступлення_z'] = standardize(days)

# 4) відношень — Цукор1: логарифм прибирає асиметрію, потім стандартизація
X['цукор1_log_z'] = standardize(np.log(CLEAN_DATA['Цукор1']))

# 5) абсолютна — Лім1: min-max із перевіркою варіативності
X['лім1_01'] = minmax(CLEAN_DATA['Лім1'])

# решта ознак варіанта
X['КТ_частка'] = CLEAN_DATA['КТ']
X['САТ_z'] = standardize(CLEAN_DATA['АТ пост (САТ)'])
X['ДАТ_z'] = standardize(CLEAN_DATA['АТ пост (ДАТ)'])

print('Асиметрія Цукор1: до логарифма', round(CLEAN_DATA['Цукор1'].skew(), 2),
      '| після', round(np.log(CLEAN_DATA['Цукор1']).skew(), 2))
print('Телефон до X не потрапив — ідентифікатор.')
X.describe().T.round(3)

Асиметрія Цукор1: до логарифма 2.95 | після 1.06
Телефон до X не потрапив — ідентифікатор.


,count,mean,std,min,25%,50%,75%,max
супутній_ІХС,121.0,0.810,0.394,0.000,1.000,1.000,1.000,1.000
супутній_ГХ,121.0,0.579,0.496,0.000,0.000,1.000,1.000,1.000
супутній_ЦД,121.0,0.240,0.429,0.000,0.000,0.000,0.000,1.000
супутній_СН,121.0,0.273,0.447,0.000,0.000,0.000,1.000,1.000
ступінь_КТ,42.0,1.476,1.110,0.000,1.000,1.000,2.000,4.000
ступінь_КТ_пропуск,121.0,0.653,0.478,0.000,0.000,1.000,1.000,1.000
поступлення_z,120.0,-0.000,1.000,-2.842,-0.602,0.409,0.749,1.253
цукор1_log_z,118.0,0.000,1.000,-1.829,-0.767,-0.077,0.529,3.546
лім1_01,117.0,0.237,0.187,0.000,0.107,0.196,0.304,1.000
КТ_частка,43.0,0.425,0.220,0.050,0.262,0.350,0.638,0.800


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [22]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [23]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [25]:
ВИПРАВЛЕННЯ = [
    (1, 'помер/жив — бінарна номінальна змінна (1 = помер, 2 = вижив); середнє для номінальної шкали недопустиме',
     'Число 1.50 (точно 1.496) — середнє міток. «Посередині між смертю і одужанням» не був жоден пацієнт; '
     'при кодуванні 0/1 те саме «середнє» стало б 0.50 з іншим змістом.',
     'Частоти й частки: померло 61 з 121 (летальність 50.4 %), вижило 60 (49.6 %).'),
    (2, 'Номінальна змінна не очищена: «ж»/«жін» і «ч»/«чол» — ті самі категорії з різним написанням',
     'nunique() = 4 рахує різні рядки, а не статі; «чотири статі» — артефакт введення, і частка «ж» 46 % занижена.',
     'Звести написання (str.strip().str[0]) і рахувати частоти: ж — 75 (62 %), ч — 46 (38 %).'),
    (3, 'Порядкова шкала: рівні впорядковано sort_index за алфавітом, написання не очищено, і з кодів пораховано середнє',
     '«Найлегший — важкий», «найважчий — середній» — це порядок літер. Середній код 5.9 залежить від числа описок '
     'і дає «легкий» лише тому, що «л» стоїть посередині алфавіту.',
     'Очистити написання, задати порядок вручну (легкий < … < критичний), звітувати частоти й медіану — «середній».'),
    (4, 'Дані непридатні: pd.to_numeric мовчки перетворив 17 рядків-інтервалів («70-80%», «до 17%») на NaN',
     'Середнє 39.6 % пораховано лише по 27 з 44 пацієнтів із КТ; викинуті записи мають більше ураження '
     '(у середньому 47 %), тож число зміщене вниз і не є «середнім обсягом ураження».',
     'Розібрати інтервали (середина /100) і рахувати по 43 значеннях: середнє 42.5 %, медіана 35 %; зазначити, що КТ є лише в 44 з 121.'),
    (5, 'Дані непридатні: у spO2 є неможливе значення 0.9 (помилка введення), а висновок про «більшість» зроблено за середнім',
     'Середнє 82.05 % тягне вниз одне хибне значення; середнє не описує «більшість» пацієнтів — для цього потрібна частка.',
     'Виправити або виключити 0.9, звітувати медіану (84 %) і частку пацієнтів із SpO2 < 90 % — це ≈ 76 %.'),
    (6, 'Інтервальна шкала: коефіцієнт варіації для дат недопустимий, бо нуль (01.01.1970) умовний',
     '0.595 % — розкид дат, поділений на відстань від 1970 року. Якщо відлік вести від 01.01.2021, вийде ≈ 30 %. '
     'Про однорідність потоку госпіталізацій це число не каже нічого.',
     'Описати розкид у днях: діапазон 25.02.2021 – 03.06.2022, ст. відхилення ≈ 113 днів, IQR, або кількість госпіталізацій по місяцях.'),
    (7, 'Номінальна × номінальна: Пірсон і «лінійний зв’язок» для статі й летальності не мають сенсу',
     'r = 0.177 — це φ, знак якого залежить від того, кого закодовано 1. «Близьке до нуля → зв’язку немає» хибно: '
     'летальність жінок 57 %, чоловіків 39 %, χ² p ≈ 0.05–0.08 — зв’язок слабкий, на межі значущості.',
     'Таблиця спряженості, χ² та коефіцієнт Крамера V (= 0.18) з p-значенням.'),
    (8, 'Нормалізація без перевірки варіативності й без урахування шкал: ФЕРИТ2 сталий, помер/жив — номінальні мітки',
     'У ФЕРИТ2 одне значення, max = min, знаменник 0 — стовпець мовчки став NaN (count = 0 у таблиці), тож '
     '«усі значення в [0, 1]» неправда; нормалізувати мітки 1/2 безглуздо.',
     'Перед нормалізацією відкинути сталі стовпці (nunique ≤ 1) і номінальні/ідентифікатори; min-max — лише для кількісних шкал.'),
    (9, 'Дані непридатні для dropna(): жоден рядок не заповнений повністю (ФЕРИТ2 — 1 значення з 121, ІЛ6-2 — 6)',
     'Після dropna() лишилося 0 рядків — «повністю заповнений набір, придатний для моделі» насправді порожній.',
     'Спершу відібрати потрібні ознаки й прибрати майже порожні стовпці, потім dropna(subset=…) або імпутація.'),
    (10, 'Витік цілі й мішанина шкал: «Дата розтину» існує лише для померлих, NaT перетворено на int64-мінімум; |r| Пірсона рахується і для номінальних кодів',
     'r = 1.00 тому, що дата розтину заповнена рівно для 61 померлого і порожня для 60 живих — це сама ціль, а не предиктор: '
     'на момент прогнозу розтину ще немає. Показники «2» (Лім2, ІЛ6-2) теж вимірюються пізніше й можуть містити витік.',
     'Виключити «Дату розтину» та ознаки, виміряні після результату; для кількісних ознак — r_pb, для категоріальних — V Крамера, '
     'важливість перевіряти на моделі з валідацією.'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,"помер/жив — бінарна номінальна змінна (1 = помер, 2 = ви...",Число 1.50 (точно 1.496) — середнє міток. «Посередині мі...,"Частоти й частки: померло 61 з 121 (летальність 50.4 %),..."
1,2,Номінальна змінна не очищена: «ж»/«жін» і «ч»/«чол» — ті...,"nunique() = 4 рахує різні рядки, а не статі; «чотири ста...",Звести написання (str.strip().str[0]) і рахувати частоти...
2,3,Порядкова шкала: рівні впорядковано sort_index за алфаві...,"«Найлегший — важкий», «найважчий — середній» — це порядо...","Очистити написання, задати порядок вручну (легкий < … < ..."
3,4,Дані непридатні: pd.to_numeric мовчки перетворив 17 рядк...,Середнє 39.6 % пораховано лише по 27 з 44 пацієнтів із К...,Розібрати інтервали (середина /100) і рахувати по 43 зна...
4,5,Дані непридатні: у spO2 є неможливе значення 0.9 (помилк...,Середнє 82.05 % тягне вниз одне хибне значення; середнє ...,"Виправити або виключити 0.9, звітувати медіану (84 %) і ..."
5,6,Інтервальна шкала: коефіцієнт варіації для дат недопусти...,"0.595 % — розкид дат, поділений на відстань від 1970 рок...","Описати розкид у днях: діапазон 25.02.2021 – 03.06.2022,..."
6,7,Номінальна × номінальна: Пірсон і «лінійний зв’язок» для...,"r = 0.177 — це φ, знак якого залежить від того, кого зак...","Таблиця спряженості, χ² та коефіцієнт Крамера V (= 0.18)..."
7,8,Нормалізація без перевірки варіативності й без урахуванн...,"У ФЕРИТ2 одне значення, max = min, знаменник 0 — стовпец...",Перед нормалізацією відкинути сталі стовпці (nunique ≤ 1...
8,9,Дані непридатні для dropna(): жоден рядок не заповнений ...,Після dropna() лишилося 0 рядків — «повністю заповнений ...,Спершу відібрати потрібні ознаки й прибрати майже порожн...
9,10,Витік цілі й мішанина шкал: «Дата розтину» існує лише дл...,"r = 1.00 тому, що дата розтину заповнена рівно для 61 по...","Виключити «Дату розтину» та ознаки, виміряні після резул..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [27]:
print('=== Розділ 1. Результат лікування ===')
print('НЕПРАВИЛЬНО  середнє міток           :', round(df['помер/жив'].mean(), 2))
outcome_counts = df['помер/жив'].map({1: 'помер', 2: 'вижив'}).value_counts()
print('ПРАВИЛЬНО    частоти й частки        :',
      ', '.join(f'{k} {v} ({100 * v / outcome_counts.sum():.1f} %)' for k, v in outcome_counts.items()))
print()

print('=== Розділ 3. Ступінь ураження за КТ ===')
alphabetical_order = df['ступінь КТ'].value_counts().sort_index()
print('НЕПРАВИЛЬНО  «найлегший … найважчий» :', alphabetical_order.index[0], '…', alphabetical_order.index[-1],
      '| середній код', round(df['ступінь КТ'].map({label: i for i, label in enumerate(alphabetical_order.index)}).mean(), 2))
severity = CLEAN_DATA['ступінь КТ']
median_severity = severity.cat.categories[int(pd.Series(severity.dropna().cat.codes).quantile(.5, interpolation='lower'))]
print('ПРАВИЛЬНО    порядок рівнів          :', ' < '.join(CT_ORDER))
print('             частоти                 :', severity.value_counts(sort=False).to_dict())
print('             медіана                 :', median_severity)
print()

print('=== Розділ 7. Стать × летальність ===')
sex = df['Стать'].astype(str).str.strip().str[0].map({'ж': 'жінки', 'ч': 'чоловіки'})
sex_code = sex.map({'чоловіки': 0, 'жінки': 1})
print('НЕПРАВИЛЬНО  Пірсон                  :', round(sex_code.corr(deceased), 3))
contingency_table = pd.crosstab(sex, df['помер/жив'].map({1: 'помер', 2: 'вижив'}))
chi2, p, *_ = stats.chi2_contingency(contingency_table, correction=False)
V = np.sqrt(chi2 / (contingency_table.values.sum() * (min(contingency_table.shape) - 1)))
print('ПРАВИЛЬНО    χ² =', round(chi2, 2), ' p =', round(p, 3), ' V Крамера =', round(V, 3))
print((100 * contingency_table['помер'] / contingency_table.sum(axis=1)).round(1).rename('летальність, %').to_string())
print()


=== Розділ 1. Результат лікування ===
НЕПРАВИЛЬНО  середнє міток           : 1.5
ПРАВИЛЬНО    частоти й частки        : помер 61 (50.4 %), вижив 60 (49.6 %)

=== Розділ 3. Ступінь ураження за КТ ===
НЕПРАВИЛЬНО  «найлегший … найважчий» : важкий … середній | середній код 5.9
ПРАВИЛЬНО    порядок рівнів          : легкий < середній < важкий < вкрай важкий < критичний
             частоти                 : {'легкий': 7, 'середній': 18, 'важкий': 10, 'вкрай важкий': 4, 'критичний': 3}
             медіана                 : середній

=== Розділ 7. Стать × летальність ===
НЕПРАВИЛЬНО  Пірсон                  : 0.177
ПРАВИЛЬНО    χ² = 3.78  p = 0.052  V Крамера = 0.177
Стать
жінки       57.3
чоловіки    39.1



---
# Етап 7. Висновки та індивідуальне питання

In [28]:
severity = CLEAN_DATA['ступінь КТ']
glucose = CLEAN_DATA['Цукор1']
ВИСНОВКИ = [
    f'«ступінь КТ» заповнено лише у {severity.notna().sum()} пацієнтів із {len(df)}, а {df["ступінь КТ"].nunique()} '
    f'різних написань зводяться до {severity.nunique()} рівнів; найчастіший і медіанний рівень — «{median_severity}» '
    f'({(severity == median_severity).sum()} пацієнтів), тож будь-які висновки про тяжкість за КТ стосуються меншості відділення.',
    f'Вік пов’язаний із летальністю: точковий бісеріальний r = {r_pb:.2f} (p = {p_pb:.0e}); померлі в середньому '
    f'на {groups.loc["помер", "mean"] - groups.loc["вижив", "mean"]:.0f} років старші ({groups.loc["помер", "mean"]:.0f} проти '
    f'{groups.loc["вижив", "mean"]:.0f}), але вік пояснює лише ≈{100 * r_pb**2:.0f} % розкиду результату.',
    f'«Цукор1» сильно скошений управо: медіана {glucose.median():.1f} ммоль/л, середнє {glucose.mean():.1f}, максимум '
    f'{glucose.max():.1f}; у {(glucose > 11.1).sum()} пацієнтів глюкоза > 11.1 ммоль/л, а ЦД серед супутніх у '
    f'{diagnosis_counts["ЦД"]} — тому для моделі стовпець логарифмуємо.',
    f'«Телефон» і «АТ пост» pandas читає як текст: перший — ідентифікатор, який у модель не йде взагалі, а другий '
    f'містить дві величини шкали відношень (медіана САТ {CLEAN_DATA["АТ пост (САТ)"].median():.0f}, ДАТ '
    f'{CLEAN_DATA["АТ пост (ДАТ)"].median():.0f} мм рт. ст.), які треба розділити перед будь-якою статистикою.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. «ступінь КТ» заповнено лише у 42 пацієнтів із 121, а 12 різних написань зводяться до 5 рівнів; найчастіший і медіанний рівень — «середній» (18 пацієнтів), тож будь-які висновки про тяжкість за КТ стосуються меншості відділення.
2. Вік пов’язаний із летальністю: точковий бісеріальний r = 0.45 (p = 2e-07); померлі в середньому на 11 років старші (70 проти 59), але вік пояснює лише ≈20 % розкиду результату.
3. «Цукор1» сильно скошений управо: медіана 6.9 ммоль/л, середнє 8.0, максимум 33.8; у 17 пацієнтів глюкоза > 11.1 ммоль/л, а ЦД серед супутніх у 29 — тому для моделі стовпець логарифмуємо.
4. «Телефон» і «АТ пост» pandas читає як текст: перший — ідентифікатор, який у модель не йде взагалі, а другий містить дві величини шкали відношень (медіана САТ 140, ДАТ 90 мм рт. ст.), які треба розділити перед будь-якою статистикою.


In [29]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()

codes = severity.dropna().cat.codes
steps = {0: 1, 1: 2, 2: 4, 3: 8, 4: 16}            # той самий порядок, інші «відстані»
mean_1 = codes.mean()
mean_2 = codes.map(steps).mean()
nearest_level_2 = CT_ORDER[int(np.argmin([abs(mean_2 - steps[i]) for i in range(5)]))]
median_1 = CT_ORDER[int(codes.quantile(.5, interpolation='lower'))]
median_2 = CT_ORDER[[k for k, v in steps.items() if v == codes.map(steps).quantile(.5, interpolation='lower')][0]]

ВІДПОВІДЬ = (
    f'«Ступінь КТ» — порядкова шкала: відомо, що легкий < середній < важкий < вкрай важкий < критичний, '
    f'але невідомо, чи «відстань» від легкого до середнього така сама, як від важкого до критичного. '
    f'Медіана використовує тільки порядок — це рівень, вище й нижче якого по половині пацієнтів, — тому '
    f'вона не залежить від того, якими числами ми позначимо рівні. Середнє ж додає й ділить коди, тобто '
    f'припускає рівні відстані, яких шкала не гарантує.\n\n'
    f'Приклад на моїх даних (ступінь заповнено у {len(codes)} пацієнтів із {len(df)}). Кодування 0,1,2,3,4: середнє '
    f'{mean_1:.2f} — «між середнім і важким», а медіана — «{median_1}». Кодування 1,2,4,8,16 зберігає той самий '
    f'порядок тяжкості, але середнє стає {mean_2:.2f}, тобто найближче до рівня «{nearest_level_2}», тоді як медіана '
    f'знову «{median_2}». Отже, «середній ступінь ураження» змінюється від довільного вибору чисел, а медіана — ні. '
    f'Окремо: у сирих даних 12 написань; якщо закодувати їх за алфавітом, середнє дорівнює '
    f'{df["ступінь КТ"].map(alphabetical_codes).mean():.2f} і взагалі не пов’язане з тяжкістю.'
)
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Чому для стовпця «ступінь КТ» медіана має сенс, а середнє — ні? Наведіть конкретний приклад із вашими даними.

«Ступінь КТ» — порядкова шкала: відомо, що легкий < середній < важкий < вкрай важкий < критичний, але невідомо, чи «відстань» від легкого до середнього така сама, як від важкого до критичного. Медіана використовує тільки порядок — це рівень, вище й нижче якого по половині пацієнтів, — тому вона не залежить від того, якими числами ми позначимо рівні. Середнє ж додає й ділить коди, тобто припускає рівні відстані, яких шкала не гарантує.

Приклад на моїх даних (ступінь заповнено у 42 пацієнтів із 121). Кодування 0,1,2,3,4: середнє 1.48 — «між середнім і важким», а медіана — «середній». Кодування 1,2,4,8,16 зберігає той самий порядок тяжкості, але середнє стає 3.88, тобто найближче до рівня «важкий», тоді як медіана знову «середній». Отже, «середній ступінь ураження» змінюється від довільного вибору чисел, а медіана — ні. Окремо: у сирих даних 12 написань; якщо закод

---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [30]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'текст, перероблено',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'код, перевірено',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'текст, перероблено',
    'етап 7 — висновки й відповідь':   'текст, перероблено',
}

# Опишіть своїми словами, що саме ви перевірили (≥ 120 символів), і лише тоді поставте True
ЩО_САМЕ_Я_ПЕРЕВІРИВ = (
    'Перевірив відповідність обраних шкал вимірювання змісту змінних і допустимість статистик для кожної шкали. '
    'Перевірив очищення написань ступеня КТ, перетворення відсоткових інтервалів на частки та розділення '
    'артеріального тиску на систолічний і діастолічний. Перевірив кодування результату лікування '
    'для точкового бісеріального коефіцієнта кореляції, порядок категорій та обробку пропусків. '
    'Також перевірив логіку стандартизації й min-max нормалізації, захист від ділення на нуль '
    'для сталих стовпців і виключення ідентифікаторів із набору ознак моделі.'
)
ПІДТВЕРДЖУЮ = True    # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [31]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 10 готова до захисту.
Корнілов Артем, ШІ-31


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.